# The notebook for fine-tuning the base model using different techniques.
(I'm broke and I don't have cool GPU for setting it up locally)

## Task: Medical QA. Make the base model act like a doctor and medical assistant.

## Brief plan:
1. Gather the data for medical SFT (making model act like a medical professional), medical MCQ prompts for GRPO (actucal knowledge about topic)
2. Fine-tune base model using medical QA SFT
3. Load fine-tuned model and fine-tune it on real knowledge (MCQ prompts) using GRPO


In [2]:
!pip install trl peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.9 MB/s eta 0:00:00


## Gathering the data

- https://huggingface.co/datasets/lavita/ChatDoctor-HealthCareMagic-100k

Great dataset and already in right format.


In [13]:
from datasets import load_dataset


dataset_checkpoints = ["lavita/ChatDoctor-HealthCareMagic-100k",
                       "medalpaca/medical_meadow_wikidoc_patient_information"]

health_care_magic_dataset = load_dataset(dataset_checkpoints[0])
medical_wikidoc_dataset = load_dataset(dataset_checkpoints[1])

In [14]:
health_care_magic_dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 112165
    })
})

In [15]:
medical_wikidoc_dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 5942
    })
})

## Preparing everything for SFT training, QLoRA

I'm using the Llama-3.1-8B-Instruct as we aiming a helpful medical assistant. Why not base?
Short answer: makes life harder for no gain + computational resources

In [19]:
import torch
from transformers import BitsAndBytesConfig


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


model_checkpoint = "meta-llama/Meta-Llama-3.1-8B-Instruct"

model = AutoModelForCausalLM.from_pretrained(model_checkpoint,
                                             quantization_config=bnb_config,
                                             device_map="auto")

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

if not tokenizer.pad_token_id:
  tokenizer.pad_token_id = model.config.eos_token_id

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
def format_chat_template_for_sft(example):
  return {
    "text": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n{example['input']}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n{example['output']}<|eot_id|>"
  }

In [16]:
health_care_magic_dataset_train = health_care_magic_dataset["train"].shuffle(seed=42).select(range(5000))

In [17]:
health_care_magic_dataset_train

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5000
})

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType


lora_config = LoraConfig(
    r=8,
    target_modules = ["q_proj", "v_proj"],
    task_type = TaskType.CAUSAL_LM,
    lora_alpha = 32,
    lora_dropout = 0.05
)

lora_model = get_peft_model(model,
                            peft_config = lora_config)

lora_model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling
import wandb


wandb.init(project="MedicalQA_QLoRA_GRPO")

training_args = SFTConfig(
    output_dir="./llama3.2-1b-healthcaremagic-qlora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="no",
    report_to="wandb",
    fp16=False,
    bf16=True
)

trainer = SFTTrainer(
    model = lora_model,
    args = training_args,
    train_dataset = health_care_magic_dataset_train,
    processing_class = tokenizer,
    formatting_func = format_chat_template_for_sft,
)

In [ ]:
trainer.train()

In [ ]:
adapter_path = "qlora-adapter"

trainer.model.save_pretrained("qlora-adapter")
tokenizer.save_pretrained("qlora-adapter")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel


base_model = AutoModelForCausalLM.from_pretrained(
    model_checkpoint,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

In [ ]:
model = model.merge_and_unload()

model.save_pretrained("lora_merged")

## GRPO training on real knowledge.

The reward function may seem a little bit weird, because using the GRPO for this task in the way I'm using it is not really efficient. In normal cases to handle such a task we use the external reward model for evaluating the responses. In my case to make things much easier I implemented a gnarly looking reward function that reflects some logic but still uses the hard-coded labels for rewarding.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    "lora_merged",
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("lora_merged")
tokenizer.pad_token_id = tokenizer.eos_token_id

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    bias="none"
)
model = get_peft_model(model, lora_config)

In [ ]:
medical_wikidoc_dataset_train = medical_wikidoc_dataset["train"].shuffle(seed=99).select(range(5000, 6000))

In [ ]:
SYSTEM = """You are a helpful medical information assistant.
Always recommend consulting a qualified healthcare professional for diagnosis and treatment.
Never recommend specific drug dosages. Acknowledge uncertainty honestly."""

def format_chat_template_for_grpo(example):
  messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user",   "content": example["input"]}
  ]
  return {
    "prompt": tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
    )
  }

In [ ]:
medical_wikidoc_dataset_train = medical_wikidoc_dataset_train.map(format_chat_template_for_grpo,
                                                                  remove_columns=medical_wikidoc_dataset_train.column_names)

In [ ]:
import re


def medical_reward(completions, **kwargs):
  rewards = []
  for completion in completions:
    text = completion[0]["content"] if isinstance(completion[0], dict) else completion[0]
    score = 0.0
    lower = text.lower()

    # length
    word_count = len(text.split())
    if 80 <= word_count <= 300:
      score += 0.3
    elif word_count < 40:
      score -= 0.4

    # referral for serious symptoms
    serious = ["chest pain", "difficulty breathing", "severe",
                "blood", "unconscious", "stroke", "emergency"]
    doctor_terms = ["doctor", "physician", "seek medical", "consult",
                    "hospital", "emergency room", "healthcare provider"]
    mentions_serious = any(t in lower for t in serious)
    suggests_doctor = any(t in lower for t in doctor_terms)

    if suggests_doctor:
      score += 0.2
    if mentions_serious and not suggests_doctor:
      score -= 0.5

    # epistemic hedging
    hedges = ["may ", "might ", "could ", "possibly", "likely",
              "in some cases", "generally", "sometimes"]
    hedge_count = sum(1 for h in hedges if h in lower)
    if hedge_count >= 2:
      score += 0.2
    elif hedge_count == 0:
      score -= 0.2

    # hard penalty for specific dosages
    if re.search(r'\d+\s*(mg|mcg|ml|tablets?|pills?)', lower):
      score -= 0.6

    rewards.append(score)
  return rewards

In [ ]:
from trl import GRPOConfig, GRPOTrainer


grpo_config = GRPOConfig(
  output_dir="./medical-grpo",
  num_train_epochs=1,
  per_device_train_batch_size=1,
  gradient_accumulation_steps=8,
  learning_rate=5e-6,
  num_generations=4,
  max_new_tokens=300,
  max_prompt_length=512,
  logging_steps=10,
  save_strategy="epoch",
  bf16=True,
  report_to="wandb",
)

trainer = GRPOTrainer(
  model=model,
  reward_funcs=medical_reward,
  args=grpo_config,
  train_dataset=medical_wikidoc_dataset_train,
  processing_class=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
trainer.model.save_pretrained("grpo-adapter")
tokenizer.save_pretrained("grpo-adapter")

from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
  "lora_merged",
  quantization_config=bnb_config,
  device_map="auto"
)
grpo_model = PeftModel.from_pretrained(base, "grpo-adapter")
grpo_model = grpo_model.merge_and_unload()
grpo_model.save_pretrained("grpo_merged")